In [ ]:
!pip install torch transformers --quiet
import torch
from transformers import GPT2Model, GPT2Config
from torch.profiler import profile, record_function, ProfilerActivity, schedule, tensorboard_trace_handler
import os, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
!nvidia-smi -L

Using device: cuda
GPU 0: Tesla T4 (UUID: GPU-34b5ea24-e212-667c-2f6a-4923c0ed16d0)


In [ ]:
config = GPT2Config(
    n_layer=6,
    n_head=8,
    n_embd=512,
    n_positions=256,
    vocab_size=50257,
)
model = GPT2Model(config).to(device)

batch_size = 8
seq_len = 128
inputs = torch.randint(0, config.vocab_size, (batch_size, seq_len), device=device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.1f}M")
print(f"Input shape: {inputs.shape}")

Model params: 44.8M
Input shape: torch.Size([8, 128])


In [ ]:
LOG_DIR = "./log_gpt2"
os.makedirs(LOG_DIR, exist_ok=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

prof_schedule = schedule(wait=1, warmup=1, active=3, repeat=1)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=prof_schedule,
    on_trace_ready=tensorboard_trace_handler(LOG_DIR),
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
) as prof:
    for step in range(10):
        with record_function("forward"):
            outputs = model(inputs).last_hidden_state
            loss = outputs.mean()
        with record_function("backward"):
            loss.backward()
        with record_function("optimizer_step"):
            optimizer.step()
            optimizer.zero_grad()
        prof.step()

print("\nTop 15 operators by CUDA time:")
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(



Top 15 operators by CUDA time:
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.07%     231.159us        34.47%     113.764ms      37.921ms       0.000us         0.00%      98.268ms      32.756ms           0 B         

In [ ]:
events = prof.key_averages()
attn_time = sum(e.device_time for e in events if "attn" in e.key.lower() or "attention" in e.key.lower())
matmul_time = sum(e.device_time for e in events if "mm" in e.key.lower() or "linear" in e.key.lower() or "addmm" in e.key.lower())
norm_time = sum(e.device_time for e in events if "norm" in e.key.lower())
total_cuda = sum(e.device_time for e in events)

print(f"Total CUDA time:    {total_cuda/1000:.2f} ms")
print(f"Attention ops:      {attn_time/1000:.2f} ms ({100*attn_time/total_cuda:.1f}%)")
print(f"Matmul/Linear ops:  {matmul_time/1000:.2f} ms ({100*matmul_time/total_cuda:.1f}%)")
print(f"LayerNorm ops:      {norm_time/1000:.2f} ms ({100*norm_time/total_cuda:.1f}%)")

Total CUDA time:    142.14 ms
Attention ops:      3.97 ms (2.8%)
Matmul/Linear ops:  4.81 ms (3.4%)
LayerNorm ops:      0.30 ms (0.2%)


In [ ]:
!find ./log_gpt2 -type f
from google.colab import files
import glob
trace_file = glob.glob("./log_gpt2/*.pt.trace.json")[0]
print(f"Downloading: {trace_file}")
files.download(trace_file)

./log_gpt2/cfa0aabf9402_879.1776750869161392422.pt.trace.json
Downloading: ./log_gpt2/cfa0aabf9402_879.1776750869161392422.pt.trace.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>